In [1]:
# importing relevant libraries
import pandas as pd
import numpy as np
import json
import os
from itertools import chain

# import boto3

from tqdm.notebook import tqdm_notebook
import time

import csv
import re

# display all rows & columns
#pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# show all outputs
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = 'all'

# Disable warnings
import warnings
warnings.filterwarnings("ignore")

**Import data for analysis**

In [2]:
# Import data
attrition_list = 'Hopeliner Copy of Sample 2022 Attrition List.xlsx'
responses = 'TTEC Interviews (Responses).xlsx'

attrition_df = pd.read_excel(attrition_list)
response_df = pd.read_excel(responses)


**Pre-processing of Dataset**

In [3]:
# Strip column names
attrition_df.columns = [x.strip() for x in attrition_df.columns]
response_df.columns = [x.strip() for x in response_df.columns]

# Response df - pre-processing
response_df.rename(columns =  {'Read Call Opening: \nHi (First Name), thank you for making time for this short interview or chat with us. As shared in our (email/ SMS) exchange, my name is (Interviewer Name). I am from e-BI Solutions. We are working with TTEC to find ways to improve their employee experience so we are grateful that you made time for this.':'Read Call Opening',
                              'Confidentiality Notice and Consent:\nPlease note that we will treat our conversation with confidentiality and only disclose information to the extent you wish to share them with TTEC. Our goal is to help them improve the employee experience through an unbiased study and analysis of the data we gather. We will ask for your sign-off on the final notes we will include in the study. Do you agree and consent to this interview/ short chat? (Share option to record video/ audio/ transcript only)\n\n(Wait for response. Proceed with questions as appropriate)': 'Confidentiality Notice and Consent',
                              }, inplace = True)

cols = ['Stated Reason for Leaving TTEC', 'Other considerations for leaving', 'Top 3 Low lights and impact to them',
'Highlights of their stay with TTEC', 'Suggestions on redesigning the experience',
'What would make you consider returning or referring friends and family members to TTEC?',
'Any questions for e-BI or TTEC?']

for col in cols:
    response_df[col] = response_df[col].astype(str)
    
# Add a new column for all reasons for leaving
response_df['All Leaving Reasons'] = response_df['Stated Reason for Leaving TTEC'] + " " + response_df['Other considerations for leaving']

## Pre-processing Descriptive Data

In [4]:
# pip install gensim
# pip install nltk
# pip install matplotlib
# pip install advertools
# Run 'python -m nltk.downloader all' on ubuntu
# pip install typing-extensions --upgrade

In [5]:
#import modules
import nltk
import gensim
import advertools as adv
import matplotlib.pyplot
import os.path
from gensim import corpora
from gensim.models import LsiModel
from nltk.tokenize import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from gensim.models.coherencemodel import CoherenceModel
import matplotlib.pyplot as plt

# Latent Dirichlet Allocation (LDA) Topic Analysis Model

**Resources:**
- [Evaluate Topic Models: Latent Dirichlet Allocation (LDA)](https://towardsdatascience.com/evaluate-topic-model-in-python-latent-dirichlet-allocation-lda-7d57484bb5d0)

- [pyLDAvis: Topic Modelling Exploration Tool That Every NLP Data Scientist Should Know](https://neptune.ai/blog/pyldavis-topic-modelling-exploration-tool-that-every-nlp-data-scientist-should-know)

In [6]:
from gensim.utils import simple_preprocess
from nltk.corpus import stopwords

# Stop words
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])

# Functions
def sent_to_words(sentences):
    for sentence in sentences:
        # deacc=True removes punctuations
        yield(gensim.utils.simple_preprocess(str(sentence), deacc=True))
        
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) 
             if word not in stop_words] for doc in texts]

def preprocess_data_indiv(doc_set):
    """
    Input  : docuemnt list
    Purpose: preprocess text (tokenize, removing stopwords, and stemming)
    Output : preprocessed text
    """
    # initialize regex tokenizer
    tokenizer = RegexpTokenizer(r'\w+')
    # create English & Tagalog stop words list
    en_stop = set(stopwords.words('english') + list(adv.stopwords['tagalog']))
    # Create p_stemmer of class PorterStemmer
    p_stemmer = PorterStemmer()
    # list for tokenized documents in loop
    texts = []
    # loop through document list
    for i in doc_set:
        # clean and tokenize document string
        raw = i.lower()
        tokens = tokenizer.tokenize(raw)
        # remove stop words from tokens
        stopped_tokens = [i for i in tokens if not i in en_stop]
        # stem tokens
        #stemmed_tokens = [p_stemmer.stem(i) for i in stopped_tokens]
        # add tokens to list
        texts.append(stopped_tokens)
    return texts


In [7]:
# NLTK Stop words
# import nltk
# nltk.download('stopwords')
from nltk.corpus import stopwords
stop_words = stopwords.words('english') + list(adv.stopwords['tagalog'])
stop_words.extend(['from', 'subject', 're', 'edu', 'use','employee','ttec','resign'])
# Define functions for stopwords, bigrams, trigrams and lemmatization
def remove_stopwords(texts):
    return [[word for word in simple_preprocess(str(doc)) if word not in stop_words] for doc in texts]
def make_bigrams(texts):
    return [bigram_mod[doc] for doc in texts]
def make_trigrams(texts):
    return [trigram_mod[bigram_mod[doc]] for doc in texts]
def lemmatization(texts, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV']):
    """https://spacy.io/api/annotation"""
    texts_out = []
    for sent in texts:
        doc = nlp(" ".join(sent)) 
        texts_out.append([token.lemma_ for token in doc if token.pos_ in allowed_postags])
    return texts_out

In [8]:
# Create a copy of the original response dataframe
analysis_df = response_df.copy()

> 1) Remove punctuations, digits and stop words from text

In [9]:
# Remove punctuations & digits
import re
from string import digits
remove_digits = str.maketrans('', '', digits) # remove digits

for col in cols:
    analysis_df[col] = analysis_df[col].apply(lambda x: re.sub(r'[^\w\s]', '', x).translate(remove_digits).strip().lower())

In [10]:
import spacy
# !python -m spacy download en_core_web_lg 
# !python -m spacy download en_core_web_sm

In [11]:
# Remove stop words
data = analysis_df['Other considerations for leaving'].to_list() # field to analyse (MANUAL INPUT)
data_words = list(sent_to_words(data)) # convert words in response to list of words

# Build the bigram and trigram models
bigram = gensim.models.Phrases(data_words, min_count=5, threshold=100) # higher threshold fewer phrases.
trigram = gensim.models.Phrases(bigram[data_words], threshold=100)
# Faster way to get a sentence clubbed as a trigram/bigram
bigram_mod = gensim.models.phrases.Phraser(bigram)
trigram_mod = gensim.models.phrases.Phraser(trigram)

data_words_nostops = remove_stopwords(data_words) # remove stop words

# Form Bigrams
data_words_bigrams = make_bigrams(data_words_nostops)

# Initialize spacy 'en' model, keeping only tagger component (for efficiency)
nlp = spacy.load("en_core_web_sm", disable=['parser', 'ner'])

# Do lemmatization keeping only noun, adj, vb, adv
data_lemmatized = lemmatization(data_words_bigrams, allowed_postags=['NOUN', 'ADJ', 'VERB', 'ADV'])

2022-12-21 09:26:18,726 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:26:18,727 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:26:18,729 | INFO | phrases.py:525 | learn_vocab | collected 1065 word types from a corpus of 885 words (unigram + bigrams) and 107 sentences
2022-12-21 09:26:18,730 | INFO | phrases.py:580 | add_vocab | using 1065 counts as vocab in Phrases<0 vocab, min_count=5, threshold=100, max_vocab_size=40000000>
2022-12-21 09:26:18,731 | INFO | phrases.py:497 | learn_vocab | collecting all words and their counts
2022-12-21 09:26:18,732 | INFO | phrases.py:502 | learn_vocab | PROGRESS: at sentence #0, processed 0 words and 0 word types
2022-12-21 09:26:18,737 | INFO | phrases.py:525 | learn_vocab | collected 1065 word types from a corpus of 885 words (unigram + bigrams) and 107 sentences
2022-12-21 09:26:18,738 | INFO | phrases.py:580 | add_vocab | using 1065 c

> 2) Create a corpus

In [12]:
import gensim.corpora as corpora

def prepare_corpus(doc_clean):
    """
    Input  : clean document
    Purpose: create term dictionary of our courpus and Converting list of documents (corpus) into Document Term Matrix
    Output : term dictionary and Document Term Matrix
    """
    # Creating the term dictionary of our courpus, where every unique term is assigned an index. dictionary = corpora.Dictionary(doc_clean)
    dictionary = corpora.Dictionary(doc_clean)
    # Converting list of documents (corpus) into Document Term Matrix using dictionary prepared above.
    doc_term_matrix = [dictionary.doc2bow(doc) for doc in doc_clean]
    # generate LDA model
    return dictionary,doc_term_matrix

In [13]:
import gensim.corpora as corpora
# Create Dictionary
id2word = corpora.Dictionary(data_lemmatized)
# Create Corpus
texts = data_lemmatized
# Term Document Frequency
corpus = [id2word.doc2bow(text) for text in texts]

2022-12-21 09:26:19,598 | INFO | dictionary.py:209 | add_documents | adding document #0 to Dictionary(0 unique tokens: [])
2022-12-21 09:26:19,600 | INFO | dictionary.py:214 | add_documents | built Dictionary(226 unique tokens: ['agent', 'attitude', 'belt', 'former', 'reason']...) from 107 documents (total 408 corpus positions)


> 3) LDA Model 

In [14]:
from pprint import pprint

In [15]:
# number of topics
num_topics = 10

# Build LDA model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                       id2word=id2word,
                                       num_topics=num_topics,
                                      random_state=3,
                                      chunksize=100,
                                       passes=10,
                                       per_word_topics=True)
# Print the Keyword in the 10 topics
pprint(lda_model.print_topics())
doc_lda = lda_model[corpus]

2022-12-21 09:26:19,608 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric alpha at 0.1
2022-12-21 09:26:19,609 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.1
2022-12-21 09:26:19,610 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:26:19,611 | INFO | ldamulticore.py:238 | update | running online LDA training, 10 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:26:19,621 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:26:19,671 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:19,677 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:21,89

2022-12-21 09:26:21,970 | INFO | ldamodel.py:1171 | show_topics | topic #8 (0.100): 0.040*"work" + 0.039*"feel" + 0.022*"salary" + 0.021*"risk" + 0.021*"public" + 0.021*"exposure" + 0.021*"travel" + 0.021*"consider" + 0.021*"malayo" + 0.021*"location"
2022-12-21 09:26:21,970 | INFO | ldamodel.py:1171 | show_topics | topic #6 (0.100): 0.032*"environment" + 0.032*"side" + 0.032*"corporate" + 0.032*"establish" + 0.032*"management" + 0.032*"empathic" + 0.032*"compensate" + 0.032*"pasture" + 0.032*"look" + 0.032*"assign"
2022-12-21 09:26:21,971 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.100): 0.076*"take" + 0.060*"care" + 0.045*"need" + 0.034*"call" + 0.034*"home" + 0.033*"mother" + 0.026*"stay" + 0.022*"family" + 0.018*"time" + 0.018*"find"
2022-12-21 09:26:21,972 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.054*"sale" + 0.028*"evaluation" + 0.028*"friendly" + 0.028*"application" + 0.028*"user" + 0.020*"none" + 0.014*"also" + 0.014*"especially" + 0.014*"ask" + 0

2022-12-21 09:26:22,040 | INFO | ldamodel.py:1171 | show_topics | topic #5 (0.100): 0.054*"sale" + 0.028*"evaluation" + 0.028*"friendly" + 0.028*"application" + 0.028*"user" + 0.015*"none" + 0.015*"also" + 0.015*"especially" + 0.015*"ask" + 0.015*"already"
2022-12-21 09:26:22,041 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.009829, rho=0.315127
2022-12-21 09:26:22,043 | INFO | ldamodel.py:822 | log_perplexity | -6.335 per-word bound, 80.7 perplexity estimate based on a held-out corpus of 7 documents with 14 words
2022-12-21 09:26:22,043 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:22,045 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:22,054 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.100): 0.030*"due" + 0.030*"problem" + 0.030*"environment" + 0.030*"offer" + 0.030*"acc

[(0,
  '0.079*"take" + 0.064*"care" + 0.049*"need" + 0.033*"call" + 0.033*"home" + '
  '0.033*"mother" + 0.032*"stay" + 0.026*"family" + 0.018*"time" + '
  '0.018*"find"'),
 (1,
  '0.039*"health" + 0.020*"problem" + 0.020*"arrangement" + 0.020*"low" + '
  '0.020*"agent" + 0.020*"wfh" + 0.020*"time" + 0.020*"hire" + 0.020*"onsite" '
  '+ 0.020*"delay"'),
 (2,
  '0.028*"feel" + 0.028*"leave" + 0.028*"personal" + 0.028*"former" + '
  '0.028*"make" + 0.028*"tl" + 0.028*"change" + 0.028*"last" + '
  '0.028*"supervisor" + 0.015*"work"'),
 (3,
  '0.050*"work" + 0.038*"go" + 0.038*"company" + 0.037*"good" + 0.027*"take" + '
  '0.026*"employee" + 0.026*"back" + 0.026*"month" + 0.026*"think" + '
  '0.026*"setup"'),
 (4,
  '0.030*"due" + 0.030*"problem" + 0.030*"environment" + 0.030*"offer" + '
  '0.030*"accept" + 0.030*"unhealthy" + 0.030*"fulltime" + 0.030*"supervisor" '
  '+ 0.030*"site" + 0.030*"become"'),
 (5,
  '0.054*"sale" + 0.028*"evaluation" + 0.028*"friendly" + 0.028*"application" '
  

> 4) Select optimal number of topics

In [16]:
# supporting function
def compute_coherence_values(corpus, dictionary, k, a, b):
    
    lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=dictionary,
                                           num_topics=k, 
                                           random_state=100,
                                           chunksize=100,
                                           passes=10,
                                           alpha=a,
                                           eta=b)
    
    coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
    
    return coherence_model_lda.get_coherence()

In [17]:
# Baseline coherence score
from gensim.models import CoherenceModel
# Compute Coherence Score
coherence_model_lda = CoherenceModel(model=lda_model, texts=data_lemmatized, dictionary=id2word, coherence='c_v')
coherence_lda = coherence_model_lda.get_coherence()
print('\nBaseline Coherence Score: ', coherence_lda)

2022-12-21 09:26:22,118 | INFO | probability_estimation.py:155 | p_boolean_sliding_window | using ParallelWordOccurrenceAccumulator(processes=7, batch_size=64) to estimate probabilities from sliding windows
2022-12-21 09:26:23,888 | INFO | text_analysis.py:530 | terminate_workers | 7 accumulators retrieved from output queue
2022-12-21 09:26:23,918 | INFO | text_analysis.py:552 | merge_accumulators | accumulated word occurrence stats for 92 virtual documents



Baseline Coherence Score:  0.4133924412488167


In [18]:
# Optimize coherence score
import tqdm
grid = {}
grid['Validation_Set'] = {}
# Topics range
min_topics = 2
max_topics = 11
step_size = 1
topics_range = range(min_topics, max_topics, step_size)

# Alpha parameter
alpha = list(np.arange(0.01, 1, 0.3))
alpha.append('symmetric')
alpha.append('asymmetric')

# Beta parameter
beta = list(np.arange(0.01, 1, 0.3))
beta.append('symmetric')

# Validation sets
num_of_docs = len(corpus)
corpus_sets = [
#     gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.25)), 
#                gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.5)), 
               gensim.utils.ClippedCorpus(corpus, int(num_of_docs*0.75)), 
               corpus]

corpus_title = [
    #'25% Corpus','50% Corpus',
    '75% Corpus', '100% Corpus']

model_results = {'Validation_Set': [],
                 'Topics': [],
                 'Alpha': [],
                 'Beta': [],
                 'Coherence': []
                }

In [19]:
# # iterate through validation corpuses
# for i in tqdm_notebook(range(len(corpus_sets))):
#     # iterate through number of topics
#     for k in topics_range:
#         # iterate through alpha values
#         for a in alpha:
#             # iterare through beta values
#             for b in beta:
#                 # get the coherence score for the given parameters
#                 cv = compute_coherence_values(corpus=corpus_sets[i], dictionary=id2word, 
#                                               k=k, a=a, b=b)
#                 # Save the model results
#                 model_results['Validation_Set'].append(corpus_title[i])
#                 model_results['Topics'].append(k)
#                 model_results['Alpha'].append(a)
#                 model_results['Beta'].append(b)
#                 model_results['Coherence'].append(cv)


In [20]:
# # Create coherence dataframe
# coherence_results = (pd.DataFrame(model_results)
#                      .sort_values(by = 'Coherence')
#                     )

In [21]:
# # Plot results - find optimal alpha & beta to find best coherence, then reconfigure alpha & beta
# plot_df = (coherence_results
#              #.query("(Alpha != 'asymmetric') & (Alpha != 'symmetric') ", engine = 'python')
#              #.query("(Validation_Set != '25% Corpus') & (Validation_Set != '50% Corpus')", engine = 'python')
#              .query("(Alpha == 'symmetric') & (Beta == 'symmetric')", engine = 'python')
#              .sort_values(by = 'Topics', ascending = False)             
#             )
# plot_df

# # Plot
# plt.plot(plot_df['Topics'].to_list(), plot_df['Coherence'].to_list())
# plt.show()

In [22]:
# # Select optimal alpha & beta
# (coherence_results
#  .query("Topics == 6")
#  .sort_values(by = 'Coherence', ascending = False)
# )

> 5) Train final model with optimized hyperparameters

In [23]:
# Train final model
lda_model = gensim.models.LdaMulticore(corpus=corpus,
                                           id2word=id2word,
                                           num_topics=5, 
                                           random_state=3,
                                           chunksize=100,
                                           passes=10,
                                           alpha=0.61,
                                           eta='symmetric')

2022-12-21 09:26:24,298 | INFO | ldamodel.py:557 | init_dir_prior | using symmetric eta at 0.2
2022-12-21 09:26:24,299 | INFO | ldamodel.py:481 | __init__ | using serial LDA version on this node
2022-12-21 09:26:24,300 | INFO | ldamulticore.py:238 | update | running online LDA training, 5 topics, 10 passes over the supplied corpus of 107 documents, updating every 700 documents, evaluating every ~107 documents, iterating 50x with a convergence threshold of 0.001000
2022-12-21 09:26:24,302 | INFO | ldamulticore.py:279 | update | training LDA model using 7 processes
2022-12-21 09:26:24,353 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:24,358 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 0, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:26,210 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.610): 0.072*"none" + 0.020*"salary" + 0.01

2022-12-21 09:26:26,311 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.610): 0.153*"none" + 0.027*"problem" + 0.019*"feel" + 0.019*"basic" + 0.019*"wfh" + 0.016*"salary" + 0.014*"health" + 0.012*"offer" + 0.012*"arrangement" + 0.011*"work"
2022-12-21 09:26:26,312 | INFO | ldamodel.py:1171 | show_topics | topic #2 (0.610): 0.057*"none" + 0.023*"low" + 0.022*"leave" + 0.020*"tl" + 0.020*"last" + 0.020*"change" + 0.020*"try" + 0.020*"letter" + 0.020*"resignation" + 0.015*"really"
2022-12-21 09:26:26,313 | INFO | ldamodel.py:1171 | show_topics | topic #3 (0.610): 0.053*"none" + 0.031*"work" + 0.028*"sale" + 0.022*"good" + 0.021*"company" + 0.020*"go" + 0.017*"environment" + 0.016*"take" + 0.015*"leave" + 0.015*"month"
2022-12-21 09:26:26,313 | INFO | ldamodel.py:1171 | show_topics | topic #4 (0.610): 0.103*"none" + 0.045*"salary" + 0.025*"company" + 0.017*"really" + 0.017*"family" + 0.016*"compare" + 0.016*"well" + 0.016*"evaluation" + 0.016*"especially" + 0.011*"work"
2022-12-21 09

2022-12-21 09:26:26,398 | INFO | ldamodel.py:1049 | do_mstep | topic diff=0.040411, rho=0.315127
2022-12-21 09:26:26,400 | INFO | ldamodel.py:822 | log_perplexity | -6.212 per-word bound, 74.1 perplexity estimate based on a held-out corpus of 7 documents with 14 words
2022-12-21 09:26:26,401 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #0 = documents up to #100/107, outstanding queue size 1
2022-12-21 09:26:26,402 | INFO | ldamulticore.py:294 | update | PROGRESS: pass 9, dispatched chunk #1 = documents up to #107/107, outstanding queue size 2
2022-12-21 09:26:26,414 | INFO | ldamodel.py:1171 | show_topics | topic #0 (0.610): 0.035*"take" + 0.029*"care" + 0.029*"personal" + 0.022*"need" + 0.022*"home" + 0.022*"agent" + 0.022*"call" + 0.022*"former" + 0.022*"supervisor" + 0.021*"family"
2022-12-21 09:26:26,415 | INFO | ldamodel.py:1171 | show_topics | topic #1 (0.610): 0.277*"none" + 0.026*"problem" + 0.018*"wfh" + 0.018*"feel" + 0.014*"basic" + 0.011*"salar

> 4) Visualizing topics

For (1), you can manually select each topic to view its top most frequent and/or “relevant” terms, using different values of the λ parameter. This can help when you’re trying to assign a human interpretable name or “meaning” to each topic.

For (2), exploring the Intertopic Distance Plot can help you learn about how topics relate to each other, including potential higher-level structure between groups of topics.

**Interpretation:**


*Bar chart*
- Each bubble represents a topic. The larger the bubble, the higher percentage of the number of responses in the corpus is about that topic.
- Blue bars represent the overall frequency of each word in the corpus. If no topic is selected, the blue bars of the most frequently used words will be displayed.
- Red bars give the estimated number of times a given term was generated by a given topic. The word with the longest red bar is the word that is used the most by the responses belonging to that topic.
- <u>*Lambda*</u> -- Decreasing the lambda parameter, increases the weight of the ratio of the frequency of word given the topic / Overall frequency of the word in the documents. Important words for the given topic moves upward.
    - Lambda = 1, top words by frequency
    - Lambda = 0, top words by relevance to topic

*Bubble plot*
- The further the bubbles are away from each other, the more different they are. 
- The larger the bubble, the more frequent is the topic in the documents.
- Distance between the topics is an approximation of semantic relationship between the topics.
- The topic which shares common words will be overlapping (closer in distance) in comparison to the non-overlapping topic.
- A good topic model will have big and non-overlapping bubbles scattered throughout the chart. As we can see from the graph, the bubbles are clustered within one place.
- A topic model with a low number of topics will have big non-overlapping bubbles, scattered throughout the chart whereas, the topic model with a high number of topics, will have many overlapping small size bubbles, clustered in the chart.


In [24]:
import pyLDAvis.gensim
import pickle 
import pyLDAvis

In [25]:
# Visualize the topics
pyLDAvis.enable_notebook()
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared = pyLDAvis.gensim.prepare(lda_model, corpus, id2word)
LDAvis_prepared

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


PreparedData(topic_coordinates=              x         y  topics  cluster       Freq
topic                                                
0      0.151109  0.058835       1        1  23.375336
3     -0.003276 -0.145732       2        1  23.008735
4     -0.073383  0.056603       3        1  20.516323
1     -0.104834  0.045370       4        1  16.944542
2      0.030383 -0.015076       5        1  16.155064, topic_info=        Term       Freq      Total Category  logprob  loglift
104     none  28.000000  28.000000  Default  30.0000  30.0000
30    salary   6.000000   6.000000  Default  29.0000  29.0000
19       low   3.000000   3.000000  Default  28.0000  28.0000
75      take   5.000000   5.000000  Default  27.0000  27.0000
189     sale   3.000000   3.000000  Default  26.0000  26.0000
..       ...        ...        ...      ...      ...      ...
55       ask   0.818570   1.989634   Topic5  -4.3885   0.9348
26    family   0.843469   4.628532   Topic5  -4.3586   0.1205
39      feel   0.818833   3.731863   Topic5  -4.3882   0.3062
4     reason   0.813934   2.640288   Topic5  -4.3942   0.6462
10   company   0.819405   5.877782   Topic5  -4.3875  -0.1474

[229 rows x 6 columns], token_table=      Topic      Freq      Term
term                           
32        5  0.759486  addition
106       1  0.507968    affect
106       3  0.507968    affect
0         1  0.760910     agent
8         2  0.396345   already
...     ...       ...       ...
80        3  0.153131      work
80        4  0.153131      work
138       2  0.759503    workno
97        4  0.839198     would
98        4  0.839139      zoom

[222 rows x 3 columns], R=30, lambda_step=0.01, plot_opts={'xlab': 'PC1', 'ylab': 'PC2'}, topic_order=[1, 4, 5, 2, 3])

# Post-modeling Exploratory Data Analysis

*Assign respondents to their topic clusters*

In [26]:
# Topic distribution for the whole document. Each element in the list is a pair of a topic’s id, and the probability that was assigned to it.
topic_dist = list(lda_model.get_document_topics(corpus))


# Classify responses to topics
cluster = []
probability = []

for i in range(0, len(topic_dist)):
    clust = [x for x,y in topic_dist[i]]
    prob = [y for x,y in topic_dist[i]]
    get_index = prob.index(max(prob))
    max_val = topic_dist[i][get_index]
    cluster.append(max_val[0])
    probability.append(max_val[1])
    
# Add classification to df
analysis_df['Topic'] = cluster
analysis_df['Topic_Probability'] = probability

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [27]:
# Map topics to Oracle ID
id_topic_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic']))
id_prob_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['Topic_Probability']))
id_rate_mapper = dict(zip(analysis_df['Oracle ID of former employee'], analysis_df['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']))

# For those with missing Oracle ID, map to names
name_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic']]
                
                     .set_index("Name").to_dict()['Topic']
)

name_mapper2 = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','Topic_Probability']]
                
                     .set_index("Name").to_dict()['Topic_Probability']
)

rating_mapper = (analysis_df[~analysis_df.applymap(np.isreal)['Oracle ID of former employee']]
               .assign(Name = lambda x: x['Name of former employee'].apply(lambda x: x.upper()))
               [['Name','How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']]
                
                     .set_index("Name").to_dict()['How would you rate your overall experience with TTEC on a scale of 1 to 5? 5 being Very Good and 1 being Not Good']
)

# Filter attrition df to contain respondents only
respondents_demog_df = (attrition_df
                         .assign(Topic1 = lambda x: x['EE Oracle Id'].map(id_topic_mapper),
                                 Prob1 = lambda x: x['EE Oracle Id'].map(id_prob_mapper),
                                 Rate1 = lambda x: x['EE Oracle Id'].map(id_rate_mapper),
                                 Name = lambda x: (x['First Name'] + " " + x['Last Name']).apply(lambda x: x.upper()),
                                 Topic = lambda x: np.where(x['Topic1'].isna(), x['Name'].map(name_mapper), x['Topic1']),
                                 Topic_Probability = lambda x: np.where(x['Prob1'].isna(), x['Name'].map(name_mapper2), x['Prob1']),
                                Rating = lambda x: np.where(x['Rate1'].isna(), x['Name'].map(rating_mapper), x['Rate1']),

                         )
                         .drop(['Topic1','Prob1','Rate1'], axis = 1)
                         .query("Topic.notna()", engine = 'python')
                         .sort_values(by = 'Topic')
                        )

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


> 1) How many respondents belong to each topic cluster & what is the average probability score?

In [28]:
# "Number of Employees per Cluster & Average Probability Score")
gen_stats = (analysis_df
             .groupby(['Topic']).agg({'Oracle ID of former employee':'nunique', 'Topic_Probability':'describe'})
             .reset_index()
            )

gen_stats

print("Number of Employees per Cluster & Average Probability Score")


/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Topic Oracle ID of former employee Topic_Probability                      \
        Oracle ID of former employee             count      mean       std   
0     0                           29              29.0  0.437508  0.262753   
1     1                           50              50.0  0.412297  0.102538   
2     2                            5               5.0  0.634117  0.247964   
3     3                           13              13.0  0.636624  0.178088   
4     4                           10              10.0  0.643820  0.201766   

                                                     
        min       25%       50%       75%       max  
0  0.200000  0.200000  0.200000  0.698881  0.856930  
1  0.378827  0.378846  0.378856  0.378867  0.879003  
2  0.312896  0.512960  0.581255  0.845293  0.918182  
3  0.342836  0.530142  0.629969  0.794119  0.893021  
4  0.389035  0.430949  0.685374  0.819393  0.886861

Number of Employees per Cluster & Average Probability Score


> 2) Tenure, Topic Probability, and Employee Count by By Attrition Stage

In [29]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median'],'Topic_Probability':['mean', 'std','median']})
 .unstack('Stage Attrition')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id            Tenure In Months             \
                       nunique                        mean              
Stage Attrition Pre Production Production   Pre Production Production   
Topic                                                                   
0.0                        5.0       24.0         6.200000  21.833333   
1.0                        6.0       44.0         5.833333  27.886364   
2.0                        NaN        5.0              NaN  23.600000   
3.0                        1.0       12.0         3.000000  28.083333   
4.0                        2.0        8.0         3.000000  22.125000   

                                                                     \
                           std                    median              
Stage Attrition Pre Production Production Pre Production Production   
Topic                                                                 
0.0                   8.408329  18.767842            2.0       20.0   
1.0                  11.839200  24.323609            1.0       20.0   
2.0                        NaN  14.587666            NaN       19.0   
3.0                        NaN  20.246025            3.0       22.5   
4.0                   1.414214  19.931579            3.0       21.0   

                Topic_Probability                                       \
                             mean                       std              
Stage Attrition    Pre Production Production Pre Production Production   
Topic                                                                    
0.0                      0.509788   0.422450       0.287550   0.261334   
1.0                      0.440821   0.408407       0.151785   0.095776   
2.0                           NaN   0.634117            NaN   0.247964   
3.0                      0.686918   0.632433            NaN   0.185336   
4.0                      0.454394   0.691177       0.082717   0.196337   

                                           
                        median             
Stage Attrition Pre Production Production  
Topic                                      
0.0                   0.652950   0.200000  
1.0                   0.378860   0.378854  
2.0                        NaN   0.581255  
3.0                   0.686918   0.613875  
4.0                   0.454394   0.756475

> 3) Tenure, Topic Probability, and Employee Count by By Job Name

In [30]:
(respondents_demog_df
 .groupby(['Topic','Job Name','Stage Attrition']).agg({'EE Oracle Id':'nunique','Tenure In Months':['mean', 'std','median']})
 .unstack('Job Name')
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


EE Oracle Id                                        \
                           nunique                                         
Job Name                     CSR I CSR II CSR III Chat Sales Associate I   
Topic Stage Attrition                                                      
0.0   Pre Production           2.0    3.0     NaN                    NaN   
      Production               2.0   20.0     2.0                    NaN   
1.0   Pre Production           1.0    5.0     NaN                    NaN   
      Production               7.0   29.0     6.0                    NaN   
2.0   Production               1.0    1.0     2.0                    NaN   
3.0   Pre Production           1.0    NaN     NaN                    NaN   
      Production               3.0    5.0     2.0                    1.0   
4.0   Pre Production           2.0    NaN     NaN                    NaN   
      Production               NaN    6.0     1.0                    NaN   

                                          Tenure In Months                     \
                                                      mean                      
Job Name              ISR I TSR I TSR III            CSR I     CSR II CSR III   
Topic Stage Attrition                                                           
0.0   Pre Production    NaN   NaN     NaN         3.500000   8.000000     NaN   
      Production        NaN   NaN     NaN        19.000000  20.950000    33.5   
1.0   Pre Production    NaN   NaN     NaN         1.000000   6.800000     NaN   
      Production        NaN   1.0     1.0        28.142857  27.034483    29.5   
2.0   Production        NaN   NaN     1.0        49.000000  12.000000    20.0   
3.0   Pre Production    NaN   NaN     NaN         3.000000        NaN     NaN   
      Production        NaN   NaN     1.0        40.666667  22.200000    18.0   
4.0   Pre Production    NaN   NaN     NaN         3.000000        NaN     NaN   
      Production        1.0   NaN     NaN              NaN  19.333333     2.0   

                                                                             \
                                                                        std   
Job Name              Chat Sales Associate I ISR I TSR I TSR III      CSR I   
Topic Stage Attrition                                                         
0.0   Pre Production                     NaN   NaN   NaN     NaN   2.121320   
      Production                         NaN   NaN   NaN     NaN   2.828427   
1.0   Pre Production                     NaN   NaN   NaN     NaN        NaN   
      Production                         NaN   NaN  56.0    13.0  19.143568   
2.0   Production                         NaN   NaN   NaN    17.0        NaN   
3.0   Pre Production                     NaN   NaN   NaN     NaN        NaN   
      Production                        40.0   NaN   NaN    28.0  23.501773   
4.0   Pre Production                     NaN   NaN   NaN     NaN   1.414214   
      Production                         NaN  59.0   NaN     NaN        NaN   

                                                                          \
                                                                           
Job Name                  CSR II    CSR III Chat Sales Associate I ISR I   
Topic Stage Attrition                                                      
0.0   Pre Production   11.269428        NaN                    NaN   NaN   
      Production       20.100995  10.606602                    NaN   NaN   
1.0   Pre Production   12.969194        NaN                    NaN   NaN   
      Production       27.288777  16.379866                    NaN   NaN   
2.0   Production             NaN   1.414214                    NaN   NaN   
3.0   Pre Production         NaN        NaN                    NaN   NaN   
      Production       24.509182   2.828427                    NaN   NaN   
4.0   Pre Production         NaN        NaN                    NaN   NaN   
      Production       13.923601       

> 4) Tenure, Topic Probability, and Employee Count by By Job Name

In [31]:
(respondents_demog_df
 .groupby(['Topic','Stage Attrition']).agg({'Rating': ['mean', 'std','median']})
 .round(1)
)

/opt/anaconda3/lib/python3.8/site-packages/ipykernel/ipkernel.py:287: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


Rating            
                        mean  std median
Topic Stage Attrition                   
0.0   Pre Production     3.8  1.0    3.5
      Production         4.3  0.7    4.0
1.0   Pre Production     4.4  0.9    5.0
      Production         4.3  0.8    4.0
2.0   Production         4.0  0.0    4.0
3.0   Pre Production     4.0  NaN    4.0
      Production         3.6  0.8    4.0
4.0   Pre Production     5.0  NaN    5.0
      Production         4.1  0.4    4.0